In [2]:
import pandas as pd
import numpy as np

In [3]:
rules_df = pd.read_csv("F:\Projects\Market Basket Analysis\data\processed\simple_rules.csv")
rules_df.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_len,consequent_len
0,Total 2% Greek Strained Yogurt with Cherry 5.3 oz,Total 2% Lowfat Greek Strained Yogurt With Blu...,0.00550,0.00636,0.00163,0.296364,46.598056,1.0,0.001595,1.412150,0.983952,0.159335,0.291860,0.276326,1,1
1,Total 2% Lowfat Greek Strained Yogurt With Blu...,Total 2% Greek Strained Yogurt with Cherry 5.3 oz,0.00636,0.00550,0.00163,0.256289,46.598056,1.0,0.001595,1.337214,0.984803,0.159335,0.252176,0.276326,1,1
2,Total 2% Greek Strained Yogurt with Cherry 5.3 oz,Total 2% Lowfat Greek Strained Yogurt with Peach,0.00550,0.00614,0.00153,0.278182,45.306485,1.0,0.001496,1.376884,0.983336,0.151335,0.273722,0.263684,1,1
3,Total 2% Lowfat Greek Strained Yogurt with Peach,Total 2% Greek Strained Yogurt with Cherry 5.3 oz,0.00614,0.00550,0.00153,0.249186,45.306485,1.0,0.001496,1.324562,0.983970,0.151335,0.245033,0.263684,1,1
4,Total 2% Lowfat Greek Strained Yogurt with Peach,Total 2% with Strawberry Lowfat Greek Strained...,0.00614,0.00939,0.00256,0.416938,44.402355,1.0,0.002502,1.698979,0.983517,0.197379,0.411411,0.344784,1,1


In [4]:
print(rules_df.columns.tolist())
print(rules_df.shape)

['antecedents', 'consequents', 'antecedent support', 'consequent support', 'support', 'confidence', 'lift', 'representativity', 'leverage', 'conviction', 'zhangs_metric', 'jaccard', 'certainty', 'kulczynski', 'antecedent_len', 'consequent_len']
(1026, 16)


In [5]:
recommendation_df = rules_df[[
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift"
]].copy()

recommendation_df.head()

,antecedents,consequents,support,confidence,lift
0,Total 2% Greek Strained Yogurt with Cherry 5.3 oz,Total 2% Lowfat Greek Strained Yogurt With Blu...,0.00163,0.296364,46.598056
1,Total 2% Lowfat Greek Strained Yogurt With Blu...,Total 2% Greek Strained Yogurt with Cherry 5.3 oz,0.00163,0.256289,46.598056
2,Total 2% Greek Strained Yogurt with Cherry 5.3 oz,Total 2% Lowfat Greek Strained Yogurt with Peach,0.00153,0.278182,45.306485
3,Total 2% Lowfat Greek Strained Yogurt with Peach,Total 2% Greek Strained Yogurt with Cherry 5.3 oz,0.00153,0.249186,45.306485
4,Total 2% Lowfat Greek Strained Yogurt with Peach,Total 2% with Strawberry Lowfat Greek Strained...,0.00256,0.416938,44.402355


In [6]:
recommendation_df = recommendation_df.sort_values(
    by=["antecedents", "confidence", "lift", "support"],
    ascending=[True, False, False, False]
).reset_index(drop=True)

recommendation_df.head(10)

,antecedents,consequents,support,confidence,lift
0,1% Lowfat Milk,Banana,0.00114,0.261468,1.807090
1,100 Calorie Per Bag Popcorn,Banana,0.00103,0.251220,1.736260
2,100% Raw Coconut Water,Bag of Organic Bananas,0.00296,0.252129,2.092361
3,100% Raw Coconut Water,Organic Hass Avocado,0.00202,0.172061,2.575383
4,100% Raw Coconut Water,Organic Strawberries,0.00161,0.137138,1.653859
5,100% Raw Coconut Water,Organic Baby Spinach,0.00146,0.124361,1.668158
6,100% Raw Coconut Water,Organic Raspberries,0.00129,0.109881,2.548846
7,100% Recycled Paper Towels,Bag of Organic Bananas,0.00203,0.221374,1.837129
8,100% Recycled Paper Towels,Organic Hass Avocado,0.00124,0.135224,2.024002
9,100% Recycled Paper Towels,Organic Baby Spinach,0.00112,0.122137,1.638329


In [7]:
recommendation_df = recommendation_df.drop_duplicates(
    subset=["antecedents", "consequents"],
    keep="first"
).reset_index(drop=True)

recommendation_df.shape

(1026, 5)

In [8]:
recommendation_dict = {}

for product, group in recommendation_df.groupby("antecedents"):
    recommendation_dict[product.lower()] = group.to_dict("records")

len(recommendation_dict)

293

In [9]:
def recommend_products(product_name, top_n=5):
    """
    Recommend products based on a single input product.
    
    Parameters:
        product_name (str): Product name entered by user
        top_n (int): Number of recommendations to return
        
    Returns:
        pd.DataFrame or str
    """
    if not isinstance(product_name, str) or product_name.strip() == "":
        return "Please enter a valid product name."
    
    product_key = product_name.strip().lower()
    
    if product_key not in recommendation_dict:
        return f"No recommendations found for '{product_name}'. Product may not exist in the rule set."
    
    recs = recommendation_dict[product_key][:top_n]
    
    return pd.DataFrame(recs)[["antecedents", "consequents", "confidence", "lift", "support"]]

In [10]:
recommend_products("Bag of Organic Bananas", top_n=10)

,antecedents,consequents,confidence,lift,support
0,Bag of Organic Bananas,Organic Hass Avocado,0.167054,2.500433,0.02013
1,Bag of Organic Bananas,Organic Strawberries,0.157593,1.900547,0.01899
2,Bag of Organic Bananas,Organic Baby Spinach,0.128299,1.720976,0.01546
3,Bag of Organic Bananas,Organic Raspberries,0.107967,2.504449,0.01301


In [11]:
available_products = set(recommendation_df["antecedents"].str.lower().unique())
len(available_products)

293

In [12]:
def find_matching_products(query, max_matches=10):
    query = query.strip().lower()
    
    matches = [p for p in available_products if query in p]
    matches = sorted(matches)
    
    return matches[:max_matches]

In [13]:
def recommend_products_smart(product_name, top_n=5):
    if not isinstance(product_name, str) or product_name.strip() == "":
        return "Please enter a valid product name."
    
    user_input = product_name.strip()
    product_key = user_input.lower()
    
    # Exact match
    if product_key in recommendation_dict:
        recs = recommendation_dict[product_key][:top_n]
        result = pd.DataFrame(recs)[["antecedents", "consequents", "confidence", "lift", "support"]]
        print(f"Top {len(result)} recommendations for '{user_input}':")
        return result
    
    # Partial match suggestions
    matches = find_matching_products(user_input)
    
    if len(matches) > 0:
        return pd.DataFrame({"suggested_products": matches})
    
    return f"No recommendations found for '{user_input}'."

In [14]:
recommend_products_smart("yogurt")

,suggested_products
0,icelandic style skyr blueberry non-fat yogurt
1,non fat raspberry yogurt
2,organic plain greek whole milk yogurt
3,organic plain whole milk yogurt
4,plain greek yogurt
5,total 0% greek yogurt
6,total 0% nonfat greek yogurt
7,total 0% nonfat plain greek yogurt
8,total 0% raspberry yogurt
9,total 2% all natural greek strained yogurt wit...


In [15]:
def recommend_from_basket(product_list, top_n=10):
    """
    Recommend products based on multiple products in a basket.
    
    Parameters:
        product_list (list): List of product names already in basket
        top_n (int): Number of final recommendations
        
    Returns:
        pd.DataFrame or str
    """
    
    if not isinstance(product_list, list) or len(product_list) == 0:
        return "Please provide a non-empty list of product names."
    
    basket_lower = [str(p).strip().lower() for p in product_list if str(p).strip() != ""]
    
    if len(basket_lower) == 0:
        return "Please provide valid product names."
    
    all_recommendations = []
    not_found_products = []
    
    for product in basket_lower:
        if product in recommendation_dict:
            recs = recommendation_dict[product]
            for rec in recs:
                all_recommendations.append(rec)
        else:
            not_found_products.append(product)
    
    if len(all_recommendations) == 0:
        return "No recommendations could be generated from the provided basket."
    
    rec_df = pd.DataFrame(all_recommendations)
    
    # Remove already existing basket items from recommendations
    rec_df = rec_df[~rec_df["consequents"].str.lower().isin(basket_lower)].copy()
    
    if rec_df.empty:
        return "No new recommendations found after excluding products already in the basket."
    
    # Aggregate recommendations from multiple input products
    final_recs = rec_df.groupby("consequents").agg(
        recommendation_count=("consequents", "count"),
        avg_confidence=("confidence", "mean"),
        max_confidence=("confidence", "max"),
        avg_lift=("lift", "mean"),
        max_lift=("lift", "max"),
        avg_support=("support", "mean")
    ).reset_index()
    
    # Sort by best ranking logic
    final_recs = final_recs.sort_values(
        by=["max_confidence", "avg_confidence", "max_lift", "recommendation_count"],
        ascending=False
    ).reset_index(drop=True)
    
    # Optional info
    if len(not_found_products) > 0:
        print("Products not found in rule base:", not_found_products)
    
    return final_recs.head(top_n)

In [16]:
recommend_from_basket(["Banana", "Organic Strawberries", "Bag of Organic Bananas"], top_n=10)

,consequents,recommendation_count,avg_confidence,max_confidence,avg_lift,max_lift,avg_support
0,Organic Hass Avocado,2,0.160589,0.167054,2.403670,2.500433,0.016455
1,Organic Baby Spinach,2,0.134639,0.140979,1.806023,1.891070,0.013575
2,Organic Raspberries,2,0.117358,0.126749,2.722286,2.940122,0.011760
3,Organic Avocado,1,0.106711,0.106711,1.957639,1.957639,0.015440


In [17]:
def show_recommendations(product_name=None, basket=None, top_n=5):
    """
    Wrapper for easy notebook usage.
    Use either product_name or basket.
    """
    if product_name is not None:
        result = recommend_products_smart(product_name, top_n=top_n)
        return result
    
    elif basket is not None:
        result = recommend_from_basket(basket, top_n=top_n)
        return result
    
    else:
        return "Provide either product_name='...' or basket=[...]"

In [18]:
show_recommendations(product_name="Banana", top_n=5)

Top 1 recommendations for 'Banana':


,antecedents,consequents,confidence,lift,support
0,Banana,Organic Avocado,0.106711,1.957639,0.01544


In [19]:
show_recommendations(
product_name="Strawberry"
)

,suggested_products
0,organic strawberry smoothie
1,strawberry preserves
2,total 2% with strawberry lowfat greek strained...
3,yokids blueberry & strawberry/vanilla yogurt


In [20]:
recommendation_df.to_csv("recommendation_base.csv", index=False)
print("Saved recommendation_base.csv")

Saved recommendation_base.csv
